# LSTM Autoencoder Anomaly Detection

## Imports

In [ ]:
import sys, pathlib
ROOT = pathlib.Path("..").resolve()
sys.path.append(str(ROOT))

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
plt.rcParams["figure.dpi"] = 120

In [ ]:
from src.models.lstm_autoencoder import LSTMAutoencoder, load_config
from src.data.lstm_autoencoder_preprocessing import load_raw_data, clean_raw_data, split_train_val, fit_scaler, scale_features, create_sequences, save_processed_data, save_scaler
from src.features.lstm_autoencoder_features import engineer_features

In [ ]:
from src.training.lstm_autoencoder_training import (
    prepare_dataloaders,
    train_lstm_autoencoder,
    resolve_device,
    save_model,
    plot_training_history,
)
from src.evaluation.lstm_autoencoder_evaluation import (
    evaluate,
    evaluate_and_plot,
)

## Configuration

In [ ]:
config = load_config(ROOT / "src" / "config" / "lstm_autoencoder_config.yaml")
data_cfg = config["data"]
train_cfg = config["training"]

timestamp_col = data_cfg["timestamp_col"]
label_col = data_cfg["label_col"]
window_size = int(data_cfg["window_size"])
stride = int(data_cfg["stride"])
val_split = float(data_cfg["val_split"])
threshold_quantile = float(train_cfg["threshold_quantile"])

In [ ]:
raw_df = load_raw_data(ROOT / data_cfg["raw_path"], timestamp_col)
clean_df = clean_raw_data(raw_df, timestamp_col, label_col)

In [ ]:
feat_df, feature_cols = engineer_features(clean_df)
processed_path = save_processed_data(
    feat_df,
    ROOT / data_cfg["processed_dir"],
    data_cfg["processed_filename"],
)

In [ ]:
anomaly_mask = feat_df[label_col].astype(int) == 1
if anomaly_mask.any():
    first_anomaly_pos = np.where(anomaly_mask.values)[0][0]
else:
    first_anomaly_pos = len(feat_df)

normal_df = feat_df.iloc[:first_anomaly_pos].copy()
if normal_df.empty:
    raise ValueError("No normal data found before the first anomaly.")

train_df, val_df = split_train_val(normal_df, val_split, timestamp_col=timestamp_col)
scaler = fit_scaler(train_df, feature_cols, label_col=label_col, normal_value=0)

eval_df = feat_df.copy()

In [ ]:
train_values = scale_features(train_df, feature_cols, scaler)
val_values   = scale_features(val_df,   feature_cols, scaler)
train_seq    = create_sequences(train_values, window_size, stride)
val_seq      = create_sequences(val_values,   window_size, stride)

eval_values  = scale_features(eval_df, feature_cols, scaler)
eval_seq     = create_sequences(eval_values, window_size, stride)

In [ ]:
train_loader, val_loader = prepare_dataloaders(
    train_seq,
    val_seq,
    train_cfg["batch_size"],
)

## Model Training

In [ ]:
device = resolve_device(train_cfg["device"])
model = LSTMAutoencoder.from_config(
    ROOT / "src" / "config" / "lstm_autoencoder_config.yaml"
).to(device)
history = train_lstm_autoencoder(model, train_loader, val_loader, train_cfg, device)

In [ ]:
save_model(model, ROOT / train_cfg["model_path"])
save_scaler(scaler, ROOT / train_cfg["scaler_path"])

## Model Evaluation and Visualization

In [ ]:
plot_training_history(history)

In [ ]:
results = evaluate(
    model=model,
    train_sequences=train_seq,        
    eval_sequences=eval_seq,         
    eval_df=eval_df,                  
    feature_cols=feature_cols,
    label_col=label_col,
    timestamp_col=timestamp_col,
    window_size=window_size,
    stride=stride,
    batch_size=train_cfg["batch_size"],
    device=device,
    threshold_quantile=0.9999,     
    label_aggregation="any",
    exclude_features=["particle_count_delta"],  
)

print("=== Sequence-level metrics ===")
for k, v in results["seq_metrics"].items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

print("\n=== Row-level metrics ===")
for k, v in results["row_metrics"].items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

In [ ]:
fig = evaluate_and_plot(
    results,
    training_history=history,
    fig_title="Wind Turbine LSTM Autoencoder — Evaluation",
    save_path="evaluation_dashboard.png",
)
plt.show()